In [ ]:
import sys
if "pyodide" in sys.modules:
    import piplite
    await piplite.install('pyb2d-jupyterlite-backend==0.4.2')

# Pyodide's b2d package references an optional module that is absent from its
# wheel. Register only the missing symbol before loading the JupyterLite backend.
import types
from pathlib import Path
import b2d

_compat_name = "b2d.testbed.backend.jupyter.async_jupyter_gui"
_compat_path = (
    Path(b2d.__file__).parent
    / "testbed/backend/jupyter/async_jupyter_gui.py"
)
if not _compat_path.exists() and _compat_name not in sys.modules:
    _compat_module = types.ModuleType(_compat_name)
    _compat_module.JupyterAsyncGui = object
    sys.modules[_compat_name] = _compat_module

# The legacy renderer is not safe inside modern Pyodide workers. Keep
# physics code runnable while suppressing unsupported debug callbacks.
import pyb2d_jupyterlite_backend.plot as _backend_plot
from pyb2d_jupyterlite_backend.async_jupyter_gui import JupyterAsyncGui as _BackendGui
from IPython.display import display

_original_render_world = _backend_plot.render_world
_original_animate_world = _backend_plot.animate_world
_original_gui_init = _BackendGui.__init__

def _render_supported_layers(*args, **kwargs):
    kwargs["flags"] = []
    return _original_render_world(*args, **kwargs)

def _plot_supported_layers(*args, **kwargs):
    display(_render_supported_layers(*args, **kwargs))

def _animate_supported_layers(*args, **kwargs):
    kwargs["flags"] = []
    return _original_animate_world(*args, **kwargs)

def _gui_without_shape_batches(self, *args, **kwargs):
    _original_gui_init(self, *args, **kwargs)
    self._debug_draw_flags = []

_backend_plot.render_world = _render_supported_layers
_backend_plot.plot_world = _plot_supported_layers
_backend_plot.animate_world = _animate_supported_layers
_BackendGui.__init__ = _gui_without_shape_batches
b2d.plot.render_world = _render_supported_layers
b2d.plot.plot_world = _plot_supported_layers
b2d.plot.animate_world = _animate_supported_layers


# Initialize each scene without starting the legacy infinite asyncio loop.
import pyb2d_jupyterlite_backend.async_jupyter_gui as _backend_gui

def _finite_start_ui(self):
    self.canvas = _backend_gui.Canvas(
        width=self.resolution[0], height=self.resolution[1]
    )
    self.out = _backend_gui.ipywidgets.Output()
    self._setup_ipywidgets_gui()
    self.make_testbed()
    self._events = []
    self._stop = True
    return None

_BackendGui.start_ui = _finite_start_ui
print("PyB2D compatibility mode: scene initialized; legacy interactive drawing is disabled.")


In [ ]:
from b2d.testbed import TestbedBase
import random
import numpy
import b2d

class ColorMixing(TestbedBase):

    name = "ColorMixing"

    def __init__(self, settings=None):
        super(ColorMixing, self).__init__(settings=settings)
        dimensions = [30, 30]

        # the outer box
        box_shape = b2d.ChainShape()
        box_shape.create_loop(
            [
                (0, 0),
                (0, dimensions[1]),
                (dimensions[0], dimensions[1]),
                (dimensions[0], 0),
            ]
        )
        box = self.world.create_static_body(position=(0, 0), shape=box_shape)

        fixtureA = b2d.fixture_def(
            shape=b2d.circle_shape(1), density=2.2, friction=0.2, restitution=0.5
        )
        body = self.world.create_dynamic_body(position=(13, 10), fixtures=fixtureA)

        pdef = b2d.particle_system_def(
            viscous_strength=0.9,
            spring_strength=0.0,
            damping_strength=0.5,
            pressure_strength=0.5,
            color_mixing_strength=0.008,
            density=2,
        )
        psystem = self.world.create_particle_system(pdef)
        psystem.radius = 0.3
        psystem.damping = 1.0

        colors = [
            (255, 0, 0, 255),
            (0, 255, 0, 255),
            (0, 0, 255, 255),
            (255, 255, 0, 255),
        ]
        posiitons = [(6, 10), (20, 10), (20, 20), (6, 20)]
        for color, pos in zip(colors, posiitons):

            shape = b2d.polygon_shape(box=(5, 5), center=pos, angle=0)
            pgDef = b2d.particle_group_def(
                flags=b2d.ParticleFlag.waterParticle
                | b2d.ParticleFlag.colorMixingParticle,
                # group_flags=b2d.ParticleGroupFlag.solidParticleGroup,
                shape=shape,
                strength=1.0,
                color=color,
            )
            group = psystem.create_particle_group(pgDef)



In [ ]:
from pyb2d_jupyterlite_backend.async_jupyter_gui import JupyterAsyncGui

s = JupyterAsyncGui.Settings()
s.resolution = [1000,500]
s.scale = 8
s.fps = 40

tb = b2d.testbed.run(ColorMixing, backend=JupyterAsyncGui, gui_settings=s)